# VQ-VAE Codebook & Class Analysis

Analyzes how codebook usage patterns correlate with diagnostic classes (AD, CN, MCI).
Includes PCA/t-SNE visualizations of both codebook histograms and continuous encoder features.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import chi2_contingency
from tqdm.auto import tqdm

from eval import load_model_from_checkpoint, get_transforms
from utils import load_items
from monai.data import Dataset

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Configuration

Set paths below before running.

In [ ]:
CHECKPOINT_PATH = "/home/ng24/projects/vqvae-smm/results/checkpoint_best.pt"  # TODO: fill in
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
DATAROOT = "/data/natalia/ADNI_registered"

SPACING = 2.0        # voxel spacing in mm (2.0 = faster, 1.0 = full res)
DOWNSAMPLE = 1.0     # extra downsampling factor applied AFTER resampling to SPACING
                     # e.g. 0.5 = halve each spatial dim, 1.0 = no change
CROP_MARGIN = 0      # voxels to crop from each edge of every spatial dim (0 = no crop)
BATCH_SIZE = 4
NUM_WORKERS = 4
MAX_SUBJECTS = None  # set to e.g. 100 to cap dataset size (None = use all)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = ["AD", "CN", "MCI"]  # sorted order matches label_map in load_items
CLASS_COLORS = {"AD": "#e74c3c", "CN": "#2ecc71", "MCI": "#3498db"}

print(f"Device: {DEVICE}")

## 2. Load Model & Data

In [ ]:
model = load_model_from_checkpoint(CHECKPOINT_PATH, device=DEVICE)
model.eval()

nb_levels = model.nb_levels
nb_entries = model.codebooks[0].n_embed
print(f"Model: {nb_levels} levels, {nb_entries} codebook entries each")

In [ ]:
from monai.transforms import (
    Compose, LoadImaged, Lambdad, EnsureChannelFirstd,
    Spacingd, Orientationd, NormalizeIntensityd,
    ResizeWithPadOrCropd, Resized, CenterSpatialCropd, ToTensord,
)
from utils import _ensure_3d_image

items = load_items(DATAROOT, CSV_PATH)

# Optionally subsample to avoid OOM
if MAX_SUBJECTS is not None and len(items) > MAX_SUBJECTS:
    rng = np.random.RandomState(42)
    idx = rng.choice(len(items), MAX_SUBJECTS, replace=False)
    idx.sort()
    items = [items[i] for i in idx]

print(f"Using {len(items)} subjects")

# ── Step 1: probe spatial size from the first image ──────────────────────
# Load + resample + orient ONE image to discover the native spatial dims
# at the requested voxel spacing (no hardcoded sizes).
probe_transforms = [
    LoadImaged(keys=["image"], image_only=True),
    Lambdad(keys=["image"], func=_ensure_3d_image),
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
]
if SPACING != 1.0:
    probe_transforms.append(
        Spacingd(keys=["image"], pixdim=(SPACING, SPACING, SPACING), mode="bilinear")
    )
probe_transforms.append(Orientationd(keys=["image"], axcodes="RAS"))
probe_result = Compose(probe_transforms)({"image": items[0]["image"]})
native_size = tuple(probe_result["image"].shape[1:])  # (D, H, W) after channel dim
print(f"Native spatial size at {SPACING}mm spacing: {native_size}")

# ── Step 2: apply crop margin ────────────────────────────────────────────
if CROP_MARGIN > 0:
    spatial_size = tuple(s - 2 * CROP_MARGIN for s in native_size)
    assert all(s > 0 for s in spatial_size), (
        f"CROP_MARGIN={CROP_MARGIN} is too large for native size {native_size}"
    )
    print(f"After cropping {CROP_MARGIN}px per edge: {spatial_size}")
else:
    spatial_size = native_size

# ── Step 3: optional downsampling ────────────────────────────────────────
if DOWNSAMPLE < 1.0:
    spatial_size = tuple(max(1, int(s * DOWNSAMPLE)) for s in spatial_size)
    print(f"After downsampling (factor {DOWNSAMPLE}): {spatial_size}")

# ── Step 4: build the full transform pipeline ────────────────────────────
transform_list = [
    LoadImaged(keys=["image"], image_only=True),
    Lambdad(keys=["image"], func=_ensure_3d_image),
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
]
if SPACING != 1.0:
    transform_list.append(
        Spacingd(keys=["image"], pixdim=(SPACING, SPACING, SPACING), mode="bilinear")
    )
transform_list.append(Orientationd(keys=["image"], axcodes="RAS"))

# Crop first (removes background edges), then downsample if requested
if CROP_MARGIN > 0:
    cropped_size = tuple(s - 2 * CROP_MARGIN for s in native_size)
    transform_list.append(
        CenterSpatialCropd(keys=["image"], roi_size=cropped_size)
    )

if DOWNSAMPLE < 1.0:
    # Resize to the downsampled target (trilinear interpolation)
    transform_list.append(
        Resized(keys=["image"], spatial_size=spatial_size, mode="trilinear")
    )
else:
    # Pad/crop to uniform size (handles minor per-subject size variations)
    transform_list.append(
        ResizeWithPadOrCropd(keys=["image"], spatial_size=spatial_size)
    )

transform_list.extend([
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    ToTensord(keys=["image"], track_meta=False),
])
transforms = Compose(transform_list)

dataset = Dataset(
    data=[{"image": it["image"]} for it in items],
    transform=transforms,
)

labels = np.array([it["label"] for it in items])
subjects = [it["subject"] for it in items]

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"Final spatial size: {spatial_size}")
print(f"Class distribution: {dict(zip(CLASS_NAMES, np.bincount(labels)))}")

## 3. Extract Codebook Indices & Continuous Features

In [ ]:
# Storage: per-level lists (level 0 = finest, level nb_levels-1 = coarsest)
all_histograms = [[] for _ in range(nb_levels)]   # codebook usage histograms
all_pooled = [[] for _ in range(nb_levels)]        # continuous pooled encoder features

with torch.no_grad():
    for batch in tqdm(loader, desc="Extracting features"):
        images = batch["image"].to(DEVICE)

        # Single forward pass: return_recon=True ensures correct decoder conditioning
        # for all codebook levels (without it, non-coarsest levels get zero-padded
        # conditioning and produce wrong indices). pool_only=True returns pooled
        # encoder features as (B, C) vectors instead of full spatial maps.
        _, diffs, encoder_pools, _, id_outputs, _ = model(images, return_recon=True, pool_only=True)

        # CRITICAL: id_outputs is coarsest-first (appended during the
        # range(nb_levels-1, ..., -1) loop), but encoder_pools is finest-first
        # (appended during sequential encoder pass). Reverse id_outputs so both
        # use the same ordering: index 0 = finest, index nb_levels-1 = coarsest.
        id_outputs = id_outputs[::-1]

        B = images.shape[0]
        for lvl in range(nb_levels):
            # Codebook index histograms
            ids = id_outputs[lvl]  # (B, D, H, W)
            for b in range(B):
                hist = torch.bincount(ids[b].reshape(-1), minlength=nb_entries)
                hist = hist.float() / hist.sum()  # normalize to frequency
                all_histograms[lvl].append(hist.cpu().numpy())

            # Pooled encoder features
            all_pooled[lvl].append(encoder_pools[lvl].cpu().numpy())

# Stack into arrays
for lvl in range(nb_levels):
    all_histograms[lvl] = np.array(all_histograms[lvl])  # (N, nb_entries)
    all_pooled[lvl] = np.concatenate(all_pooled[lvl], axis=0)  # (N, C)

N = all_histograms[0].shape[0]
print(f"Extracted features for {N} subjects")
print(f"Level ordering: 0 = finest resolution, {nb_levels-1} = coarsest resolution\n")
for lvl in range(nb_levels):
    used = (all_histograms[lvl].sum(axis=0) > 0).sum()
    print(f"  Level {lvl}: histograms {all_histograms[lvl].shape}, pooled {all_pooled[lvl].shape}, "
          f"codes used across dataset: {used}/{nb_entries}")

## 4. Codebook Usage by Class

In [ ]:
for lvl in range(nb_levels):
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f"Level {lvl} — Codebook Usage by Class", fontsize=14)

    # --- Mean usage histogram per class ---
    ax = axes[0]
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        mean_usage = all_histograms[lvl][mask].mean(axis=0)
        ax.bar(
            np.arange(nb_entries), mean_usage, alpha=0.5,
            label=cls_name, color=CLASS_COLORS[cls_name],
        )
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mean frequency")
    ax.set_title("Mean codebook usage")
    ax.legend()

    # --- Heatmap: classes x entries ---
    ax = axes[1]
    usage_matrix = np.zeros((len(CLASS_NAMES), nb_entries))
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        usage_matrix[cls_idx] = all_histograms[lvl][labels == cls_idx].mean(axis=0)
    sns.heatmap(
        usage_matrix, ax=ax, cmap="viridis",
        yticklabels=CLASS_NAMES, xticklabels=False,
    )
    ax.set_xlabel("Codebook entry")
    ax.set_title("Usage heatmap (class x entry)")

    plt.tight_layout()
    plt.show()

In [ ]:
TOP_N = 10  # number of top codes to show per class

for lvl in range(nb_levels):
    print(f"{'=' * 70}")
    print(f"LEVEL {lvl}")
    print(f"{'=' * 70}")

    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        mean_usage = all_histograms[lvl][mask].mean(axis=0)
        top_indices = np.argsort(mean_usage)[::-1][:TOP_N]

        print(f"\n  {cls_name} (n={mask.sum()}) — top {TOP_N} most-used codes:")
        print(f"  {'Code':>6s}  {'Freq':>8s}  {'Bar'}")
        print(f"  {'─' * 6}  {'─' * 8}  {'─' * 30}")
        for idx in top_indices:
            freq = mean_usage[idx]
            bar = "█" * int(freq * 500)  # scale for display
            print(f"  {idx:6d}  {freq:8.4f}  {bar}")

    # Also show codes with biggest difference between classes
    usage_per_class = np.zeros((len(CLASS_NAMES), nb_entries))
    for cls_idx in range(len(CLASS_NAMES)):
        usage_per_class[cls_idx] = all_histograms[lvl][labels == cls_idx].mean(axis=0)

    # Which class uses each code the most?
    dominant_class = np.argmax(usage_per_class, axis=0)
    max_diff = usage_per_class.max(axis=0) - usage_per_class.min(axis=0)
    top_diff = np.argsort(max_diff)[::-1][:TOP_N]

    print(f"\n  Codes with largest cross-class difference:")
    print(f"  {'Code':>6s}  {'Dominant':>8s}  {'MaxFreq':>8s}  {'MinFreq':>8s}  {'Diff':>8s}")
    print(f"  {'─' * 6}  {'─' * 8}  {'─' * 8}  {'─' * 8}  {'─' * 8}")
    for idx in top_diff:
        dom = CLASS_NAMES[dominant_class[idx]]
        mx = usage_per_class[:, idx].max()
        mn = usage_per_class[:, idx].min()
        print(f"  {idx:6d}  {dom:>8s}  {mx:8.4f}  {mn:8.4f}  {mx - mn:8.4f}")
    print()

## 5. Most Discriminative Codes (Chi-squared)

In [ ]:
TOP_K = 20

for lvl in range(nb_levels):
    chi2_scores = np.zeros(nb_entries)
    for code_idx in range(nb_entries):
        # Bin usage into quartiles for chi-squared
        usage = all_histograms[lvl][:, code_idx]
        bins = np.quantile(usage[usage > 0], [0.33, 0.66]) if (usage > 0).sum() > 10 else None
        if bins is None or len(np.unique(bins)) < 2:
            continue
        digitized = np.digitize(usage, bins)
        contingency = pd.crosstab(digitized, labels)
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            chi2, p, _, _ = chi2_contingency(contingency)
            chi2_scores[code_idx] = chi2

    top_codes = np.argsort(chi2_scores)[::-1][:TOP_K]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(TOP_K), chi2_scores[top_codes], color="steelblue")
    ax.set_xticks(range(TOP_K))
    ax.set_xticklabels(top_codes, rotation=45)
    ax.set_xlabel("Codebook entry index")
    ax.set_ylabel("Chi-squared statistic")
    ax.set_title(f"Level {lvl} — Top {TOP_K} most class-discriminative codes")
    plt.tight_layout()
    plt.show()

## 6. Mutual Information: Codebook Entry vs Class

In [ ]:
for lvl in range(nb_levels):
    mi_scores = mutual_info_classif(
        all_histograms[lvl], labels, discrete_features=False, random_state=42,
    )

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(np.arange(nb_entries), mi_scores, color="darkorange", width=1.0)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mutual information (nats)")
    ax.set_title(f"Level {lvl} — MI between codebook entry usage and class label")
    plt.tight_layout()
    plt.show()

    top10 = np.argsort(mi_scores)[::-1][:10]
    print(f"Level {lvl} top-10 MI codes: {top10.tolist()}")
    print(f"  MI values: {mi_scores[top10].round(4).tolist()}")

## 7. PCA & t-SNE of Codebook Usage Histograms

In [ ]:
def scatter_by_class(ax, coords, labels, class_names, class_colors, title):
    for cls_idx, cls_name in enumerate(class_names):
        mask = labels == cls_idx
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c=class_colors[cls_name], label=cls_name,
            alpha=0.6, s=15, edgecolors="none",
        )
    ax.legend(markerscale=2)
    ax.set_title(title)


for lvl in range(nb_levels):
    X = all_histograms[lvl]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Level {lvl} — Codebook Histogram Embeddings", fontsize=14)

    scatter_by_class(
        axes[0], X_pca, labels, CLASS_NAMES, CLASS_COLORS,
        f"PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})",
    )
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")

    scatter_by_class(
        axes[1], X_tsne, labels, CLASS_NAMES, CLASS_COLORS, "t-SNE",
    )
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.show()

## 8. PCA & t-SNE of Continuous Encoder Features

In [ ]:
for lvl in range(nb_levels):
    X = all_pooled[lvl]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Level {lvl} — Continuous Encoder Features", fontsize=14)

    scatter_by_class(
        axes[0], X_pca, labels, CLASS_NAMES, CLASS_COLORS,
        f"PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})",
    )
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")

    scatter_by_class(
        axes[1], X_tsne, labels, CLASS_NAMES, CLASS_COLORS, "t-SNE",
    )
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.show()

## 9. Combined Multi-Level Feature Analysis

Concatenate features across all levels for a joint view.

In [ ]:
# Concatenate histograms across levels
X_hist_all = np.concatenate(all_histograms, axis=1)
X_pool_all = np.concatenate(all_pooled, axis=1)

for name, X in [("Codebook histograms (all levels)", X_hist_all),
                ("Pooled features (all levels)", X_pool_all)]:
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(name, fontsize=14)

    scatter_by_class(
        axes[0], X_pca, labels, CLASS_NAMES, CLASS_COLORS,
        f"PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})",
    )
    scatter_by_class(axes[1], X_tsne, labels, CLASS_NAMES, CLASS_COLORS, "t-SNE")

    plt.tight_layout()
    plt.show()

## 10. Codebook Vector Replacement & Reconstruction

Replace a specific codebook entry at a given level with another entry, decode
back to an image, and compare with the original reconstruction.

In [ ]:
import torch.nn.functional as F


@torch.no_grad()
def decode_from_indices(model, code_indices):
    """Decode a list of codebook index tensors back to an image (3D-safe).

    Args:
        model: VQVAE model.
        code_indices: list of LongTensors, one per level.
            Level ordering must match model convention (level 0 = finest).
            Each tensor has shape (B, D_l, H_l, W_l).

    Returns:
        Reconstructed image tensor (B, C, D, H, W).
    """
    decoder_outputs = []
    code_outputs = []
    upscale_counts = []

    for l in range(model.nb_levels - 1, -1, -1):
        codebook = model.codebooks[l]
        decoder = model.decoders[l]

        # embed_code -> (B, D, H, W, embed_dim), permute to (B, embed_dim, D, H, W)
        code_q = codebook.embed_code(code_indices[l]).permute(0, 4, 1, 2, 3)

        # Upscale previous code outputs
        upscaled_codes = []
        target_size = code_q.shape[2:]
        for i, c in enumerate(code_outputs):
            upscaled = model.upscalers[i](c, upscale_counts[i])
            if upscaled.shape[2:] != target_size:
                upscaled = F.interpolate(
                    upscaled, size=target_size, mode="trilinear", align_corners=False,
                )
            upscaled_codes.append(upscaled)
        code_outputs = upscaled_codes
        upscale_counts = [u + 1 for u in upscale_counts]

        decoder_in = torch.cat([code_q, *code_outputs], dim=1)
        decoder_outputs.append(decoder(decoder_in))

        code_outputs.append(code_q)
        upscale_counts.append(0)

    return decoder_outputs[-1]

In [ ]:
# ── Configuration ────────────────────────────────────────────────
SUBJECT_IDX = 0            # index into `items` list
TARGET_LEVEL = 2           # codebook level to modify (0=finest, nb_levels-1=coarsest)
OLD_CODE = 10              # codebook entry index to replace
NEW_CODE = 50              # replacement codebook entry index
SLICE_AXIS = 2             # 0=sagittal, 1=coronal, 2=axial

# ── Encode the subject ──────────────────────────────────────────
sample = dataset[SUBJECT_IDX]
image = sample["image"].unsqueeze(0).to(DEVICE)  # (1, C, D, H, W)

with torch.no_grad():
    recon_orig, _, _, _, id_outputs_raw = model(image, return_recon=True)

# id_outputs from the forward pass is coarsest-first (appended during the
# range(nb_levels-1, ..., -1) loop).  Reverse so index 0 = level 0 (finest).
id_outputs = id_outputs_raw[::-1]

# ── Diagnostics: how different are the two code embeddings? ─────
codebook = model.codebooks[TARGET_LEVEL]
emb_old = codebook.embed[:, OLD_CODE]   # (embed_dim,)
emb_new = codebook.embed[:, NEW_CODE]   # (embed_dim,)
l2_dist = (emb_old - emb_new).norm().item()
cos_sim = F.cosine_similarity(emb_old.unsqueeze(0), emb_new.unsqueeze(0)).item()
print(f"Embedding comparison (code {OLD_CODE} vs {NEW_CODE}):")
print(f"  L2 distance:       {l2_dist:.4f}")
print(f"  Cosine similarity: {cos_sim:.4f}")

# For context, show distribution of all pairwise distances
all_embeds = codebook.embed.T  # (nb_entries, embed_dim)
sample_idx = torch.randint(0, all_embeds.shape[0], (200,))
sample_embeds = all_embeds[sample_idx]
pairwise = torch.cdist(sample_embeds.unsqueeze(0), sample_embeds.unsqueeze(0)).squeeze()
mask = torch.triu(torch.ones_like(pairwise, dtype=torch.bool), diagonal=1)
print(f"  Pairwise L2 stats (sample of 200 codes): "
      f"mean={pairwise[mask].mean():.4f}, std={pairwise[mask].std():.4f}, "
      f"min={pairwise[mask].min():.4f}, max={pairwise[mask].max():.4f}")

# ── Replace codes ───────────────────────────────────────────────
modified_ids = [ids.clone() for ids in id_outputs]
n_replaced = (modified_ids[TARGET_LEVEL] == OLD_CODE).sum().item()
modified_ids[TARGET_LEVEL][modified_ids[TARGET_LEVEL] == OLD_CODE] = NEW_CODE
print(f"\nLevel {TARGET_LEVEL}: replaced {n_replaced} voxels from code {OLD_CODE} → {NEW_CODE}")
print(f"  Code map shapes: {[tuple(ids.shape) for ids in id_outputs]}")

if n_replaced == 0:
    print("⚠️  No voxels to replace! Try a different OLD_CODE that actually appears in this subject's code map.")
    # Show which codes are actually used at this level
    unique_codes = id_outputs[TARGET_LEVEL].unique().cpu().numpy()
    print(f"  Codes present at level {TARGET_LEVEL}: {sorted(unique_codes.tolist())}")

# ── Decode original & modified ──────────────────────────────────
with torch.no_grad():
    recon_orig_from_codes = decode_from_indices(model, id_outputs)
    recon_modified = decode_from_indices(model, modified_ids)

# Interpolate to match input spatial size if needed
input_shape = image.shape[2:]
if recon_orig_from_codes.shape[2:] != input_shape:
    recon_orig_from_codes = F.interpolate(
        recon_orig_from_codes, size=input_shape, mode="trilinear", align_corners=False,
    )
if recon_modified.shape[2:] != input_shape:
    recon_modified = F.interpolate(
        recon_modified, size=input_shape, mode="trilinear", align_corners=False,
    )

# ── Difference statistics ───────────────────────────────────────
diff = (recon_modified - recon_orig_from_codes).abs()
print(f"\nReconstruction difference stats:")
print(f"  min={diff.min().item():.6f}, max={diff.max().item():.6f}, "
      f"mean={diff.mean().item():.6f}, std={diff.std().item():.6f}")
print(f"  Original recon range: [{recon_orig_from_codes.min().item():.4f}, {recon_orig_from_codes.max().item():.4f}]")
print(f"  Relative max diff:    {diff.max().item() / (recon_orig_from_codes.abs().max().item() + 1e-8):.4%}")

# ── Visualize ───────────────────────────────────────────────────
def get_mid_slice(vol, axis):
    """Get the middle slice and quarter slices along a given axis from a (C, D, H, W) tensor."""
    idx_mid = vol.shape[axis + 1] // 2  # +1 to skip channel dim
    idx_q1 = vol.shape[axis + 1] // 4
    idx_q3 = 3 * vol.shape[axis + 1] // 4
    return (
        vol[0].select(axis, idx_mid).cpu().numpy(),
        vol[0].select(axis, idx_q1).cpu().numpy(),
        vol[0].select(axis, idx_q3).cpu().numpy(),
    )

# Compute consistent vmin/vmax across original and modified for fair comparison
all_recon = torch.cat([recon_orig_from_codes, recon_modified], dim=0)
vmin_recon = all_recon.min().item()
vmax_recon = all_recon.max().item()

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
titles = ["Original input", "Decoded (original codes)", "Decoded (modified codes)", "Difference (amplified)"]
volumes = [image, recon_orig_from_codes, recon_modified, diff]
slice_labels = ["mid", "Q1", "Q3"]

for i, (vol, title) in enumerate(zip(volumes, titles)):
    slices = get_mid_slice(vol[0], SLICE_AXIS)
    for j in range(3):
        if "difference" in title.lower():
            # Amplify diff: use vmax=diff.max() so the colormap fills the actual range
            diff_max = diff.max().item()
            if diff_max < 1e-8:
                diff_max = 1.0  # avoid zero range
            im = axes[j, i].imshow(slices[j], cmap="hot", origin="lower", vmin=0, vmax=diff_max)
            if j == 0:
                plt.colorbar(im, ax=axes[j, i], fraction=0.046, pad=0.04)
        elif "input" in title.lower():
            axes[j, i].imshow(slices[j], cmap="gray", origin="lower")
        else:
            # Use consistent scale for original vs modified so differences are visible
            axes[j, i].imshow(slices[j], cmap="gray", origin="lower", vmin=vmin_recon, vmax=vmax_recon)
        axes[j, i].set_title(f"{title} ({slice_labels[j]})")
        axes[j, i].axis("off")

subject_name = subjects[SUBJECT_IDX] if SUBJECT_IDX < len(subjects) else f"#{SUBJECT_IDX}"
fig.suptitle(
    f"Subject {subject_name} — Level {TARGET_LEVEL}, code {OLD_CODE} → {NEW_CODE} "
    f"({n_replaced} voxels replaced)\n"
    f"Embedding L2 dist={l2_dist:.4f}, cos_sim={cos_sim:.4f}, max_diff={diff.max().item():.6f}",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## 11. Feature Map Extraction & Analysis

Visualize encoder feature maps at each level to understand what spatial patterns
the model learns. Includes per-channel activation maps, class-wise activation
statistics, and top-activated channel analysis.

In [ ]:
# Extract full spatial encoder feature maps for a few subjects per class
# NOTE: model.forward() sets encoder_outputs[l] = None after consuming each level,
# so we run the encoder stack directly to get the spatial maps.
N_PER_CLASS = 3  # subjects per class to visualize
SLICE_AXIS_FM = 2  # 0=sagittal, 1=coronal, 2=axial

# Pick subjects: first N_PER_CLASS from each class
selected_indices = []
for cls_idx in range(len(CLASS_NAMES)):
    cls_mask = np.where(labels == cls_idx)[0]
    selected_indices.extend(cls_mask[:N_PER_CLASS].tolist())

# Extract spatial feature maps by running encoders directly
feature_maps = {idx: [] for idx in selected_indices}  # idx -> list of (C, D, H, W) per level

with torch.no_grad():
    for idx in tqdm(selected_indices, desc="Extracting feature maps"):
        sample = dataset[idx]
        image = sample["image"].unsqueeze(0).to(DEVICE)

        # Run encoder stack manually (mirrors the encoder loop in model.forward)
        encoder_outputs = []
        for enc in model.encoders:
            if len(encoder_outputs):
                encoder_outputs.append(enc(encoder_outputs[-1]))
            else:
                encoder_outputs.append(enc(image))

        for lvl in range(nb_levels):
            feature_maps[idx].append(encoder_outputs[lvl][0].cpu())  # (C, D, H, W)

print(f"Extracted spatial feature maps for {len(selected_indices)} subjects")
for lvl in range(nb_levels):
    sample_feat = feature_maps[selected_indices[0]][lvl]
    print(f"  Level {lvl}: {tuple(sample_feat.shape)}")

### 11a. Per-Channel Activation Maps

Show the top-K most activated channels (by mean activation) for one subject per class at each encoder level.

In [ ]:
TOP_K_CHANNELS = 8  # number of top channels to display

for lvl in range(nb_levels):
    # Use subjects we already extracted feature maps for (from the extraction cell above)
    # Pick the first extracted subject per class
    show_indices = []
    for c in range(len(CLASS_NAMES)):
        for idx in selected_indices:
            if labels[idx] == c:
                show_indices.append(idx)
                break

    if not show_indices:
        print(f"Level {lvl}: no feature maps available — run the extraction cell first")
        continue

    n_rows = len(show_indices)
    fig, axes = plt.subplots(
        n_rows, TOP_K_CHANNELS,
        figsize=(2.5 * TOP_K_CHANNELS, 3 * n_rows),
    )
    if n_rows == 1:
        axes = axes[np.newaxis, :]  # ensure 2D indexing
    fig.suptitle(f"Level {lvl} — Top-{TOP_K_CHANNELS} Activated Channels (mid-axial slice)", fontsize=14)

    for row, idx in enumerate(show_indices):
        feat = feature_maps[idx][lvl]  # (C, D, H, W)

        # Mean activation per channel → pick the most activated ones
        chan_means = feat.mean(dim=(1, 2, 3))  # (C,)
        top_chans = torch.argsort(chan_means, descending=True)[:TOP_K_CHANNELS]

        mid = feat.shape[1 + SLICE_AXIS_FM] // 2  # mid slice along chosen axis
        for col, ch in enumerate(top_chans):
            slc = feat[ch].select(SLICE_AXIS_FM, mid).numpy()
            ax = axes[row, col]
            im = ax.imshow(slc, cmap="inferno", origin="lower")
            ax.set_title(f"ch{ch.item()} ({chan_means[ch]:.2f})", fontsize=9)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(CLASS_NAMES[labels[idx]], fontsize=12,
                              rotation=0, labelpad=30)

    plt.tight_layout()
    plt.show()

### 11b. Channel Activation Distributions by Class

Compare the distribution of mean channel activations across diagnostic classes.
Channels where distributions diverge are potentially class-discriminative.

In [ ]:
# Compute per-channel mean activation for all subjects (using the pooled features from Section 3)
# all_pooled[lvl] has shape (N, C) — each entry is the global-average-pooled encoder output

from scipy.stats import kruskal

for lvl in range(nb_levels):
    X = all_pooled[lvl]  # (N, C)
    n_channels = X.shape[1]

    # Kruskal-Wallis test per channel: non-parametric test for class differences
    kw_stats = np.zeros(n_channels)
    kw_pvals = np.zeros(n_channels)
    for ch in range(n_channels):
        groups = [X[labels == c, ch] for c in range(len(CLASS_NAMES))]
        if all(len(g) > 1 for g in groups):
            stat, pval = kruskal(*groups)
            kw_stats[ch] = stat
            kw_pvals[ch] = pval

    # Top discriminative channels by Kruskal-Wallis statistic
    top_disc = np.argsort(kw_stats)[::-1][:16]

    fig, axes = plt.subplots(4, 4, figsize=(16, 12))
    fig.suptitle(
        f"Level {lvl} — Top-16 Class-Discriminative Channels (Kruskal-Wallis)",
        fontsize=14,
    )
    for ax_idx, ch in enumerate(top_disc):
        ax = axes[ax_idx // 4, ax_idx % 4]
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            vals = X[labels == cls_idx, ch]
            ax.hist(vals, bins=25, alpha=0.5, label=cls_name,
                    color=CLASS_COLORS[cls_name], density=True)
        ax.set_title(f"ch{ch} (H={kw_stats[ch]:.1f}, p={kw_pvals[ch]:.1e})", fontsize=9)
        if ax_idx == 0:
            ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Summary table
    print(f"Level {lvl} — Top-10 most discriminative channels:")
    print(f"  {'Channel':>8s}  {'KW stat':>10s}  {'p-value':>12s}  ", end="")
    print("  ".join(f"{'Mean ' + c:>10s}" for c in CLASS_NAMES))
    for ch in top_disc[:10]:
        means = [X[labels == c, ch].mean() for c in range(len(CLASS_NAMES))]
        print(f"  {ch:8d}  {kw_stats[ch]:10.2f}  {kw_pvals[ch]:12.2e}  ", end="")
        print("  ".join(f"{m:10.4f}" for m in means))
    print()

### 11c. Spatial Activation Difference Maps

For the most discriminative channels, show the mean spatial activation map per class
and the AD-vs-CN difference map to highlight regions where feature representations diverge.

In [ ]:
# Accumulate spatial feature maps across all subjects for class-mean maps
# We run encoders directly (model.forward nullifies encoder_outputs after use)

TOP_DISC_CHANNELS = 4  # number of discriminative channels to visualize spatially

for lvl in range(nb_levels):
    # Identify top discriminative channels from pooled features (reuse kw_stats logic)
    X = all_pooled[lvl]
    n_channels = X.shape[1]
    kw_stats_lvl = np.zeros(n_channels)
    for ch in range(n_channels):
        groups = [X[labels == c, ch] for c in range(len(CLASS_NAMES))]
        if all(len(g) > 1 for g in groups):
            stat, _ = kruskal(*groups)
            kw_stats_lvl[ch] = stat
    # .copy() is critical: [::-1] creates a view with negative strides,
    # which PyTorch cannot use as a tensor index
    top_chans = np.argsort(kw_stats_lvl)[::-1][:TOP_DISC_CHANNELS].copy()

    # Accumulate mean spatial maps per class for the top channels
    class_sum = {c: None for c in range(len(CLASS_NAMES))}
    class_count = {c: 0 for c in range(len(CLASS_NAMES))}

    with torch.no_grad():
        for batch_start in tqdm(range(0, len(dataset), BATCH_SIZE),
                                desc=f"Level {lvl} spatial maps"):
            batch_end = min(batch_start + BATCH_SIZE, len(dataset))
            batch_images = torch.stack(
                [dataset[i]["image"] for i in range(batch_start, batch_end)]
            ).to(DEVICE)

            # Run encoder stack directly to get spatial feature maps
            encoder_outputs = []
            for enc in model.encoders:
                if len(encoder_outputs):
                    encoder_outputs.append(enc(encoder_outputs[-1]))
                else:
                    encoder_outputs.append(enc(batch_images))

            feat = encoder_outputs[lvl]  # (B, C, D, H, W)

            # Only keep the top discriminative channels
            feat_sel = feat[:, top_chans].cpu().float()  # (B, TOP_DISC_CHANNELS, D, H, W)

            for b in range(feat_sel.shape[0]):
                cls = labels[batch_start + b]
                if class_sum[cls] is None:
                    class_sum[cls] = torch.zeros_like(feat_sel[b])
                class_sum[cls] += feat_sel[b]
                class_count[cls] += 1

    # Compute class means (only for classes that have subjects)
    class_means = {}
    for c in range(len(CLASS_NAMES)):
        if class_count[c] > 0:
            class_means[c] = (class_sum[c] / class_count[c]).numpy()

    # Skip if any class is missing
    if len(class_means) < len(CLASS_NAMES):
        missing = [CLASS_NAMES[c] for c in range(len(CLASS_NAMES)) if c not in class_means]
        print(f"Level {lvl}: skipping — no subjects for classes: {missing}")
        continue

    # Visualize: rows = channels, columns = classes + AD-CN difference
    n_cols = len(CLASS_NAMES) + 1  # +1 for difference map
    fig, axes = plt.subplots(
        TOP_DISC_CHANNELS, n_cols,
        figsize=(4 * n_cols, 3.5 * TOP_DISC_CHANNELS),
    )
    if TOP_DISC_CHANNELS == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(f"Level {lvl} — Mean Spatial Activations (mid-axial slice)", fontsize=14)

    for row, ch_global in enumerate(top_chans):
        # Get mid slice for each class
        class_slices = {}
        for c in range(len(CLASS_NAMES)):
            vol = class_means[c][row]  # (D, H, W) — row indexes into selected channels
            mid = vol.shape[SLICE_AXIS_FM] // 2
            class_slices[c] = np.take(vol, mid, axis=SLICE_AXIS_FM)

        # Common colorscale across classes
        vmin = min(s.min() for s in class_slices.values())
        vmax = max(s.max() for s in class_slices.values())

        for col, c in enumerate(range(len(CLASS_NAMES))):
            ax = axes[row, col]
            im = ax.imshow(class_slices[c], cmap="inferno", origin="lower",
                           vmin=vmin, vmax=vmax)
            ax.set_title(f"{CLASS_NAMES[c]}", fontsize=10)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"ch{ch_global}\n(H={kw_stats_lvl[ch_global]:.1f})",
                              fontsize=10, rotation=0, labelpad=50)

        # AD - CN difference map
        ad_idx = CLASS_NAMES.index("AD")
        cn_idx = CLASS_NAMES.index("CN")
        diff_map = class_slices[ad_idx] - class_slices[cn_idx]
        ax = axes[row, -1]
        abs_max = max(abs(diff_map.min()), abs(diff_map.max()))
        if abs_max < 1e-8:
            abs_max = 1.0
        im = ax.imshow(diff_map, cmap="RdBu_r", origin="lower",
                       vmin=-abs_max, vmax=abs_max)
        ax.set_title("AD - CN", fontsize=10)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

### 11d. Feature Map Summary Statistics Across Levels

Compare activation magnitude, sparsity, and variance across encoder levels to understand
what each level captures (fine texture vs coarse structure).

In [ ]:
# Compute summary statistics from pooled features (already extracted)
stats_data = []

for lvl in range(nb_levels):
    X = all_pooled[lvl]  # (N, C)
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        X_cls = X[mask]
        stats_data.append({
            "Level": lvl,
            "Class": cls_name,
            "Mean activation": X_cls.mean(),
            "Std activation": X_cls.std(),
            "Sparsity (% near-zero)": (np.abs(X_cls) < 0.01).mean() * 100,
            "Max activation": X_cls.max(),
            "Active channels (>0.1)": (np.abs(X_cls).mean(axis=0) > 0.1).sum(),
        })

stats_df = pd.DataFrame(stats_data)
print(stats_df.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ["Mean activation", "Std activation", "Sparsity (% near-zero)"]):
    for cls_name in CLASS_NAMES:
        subset = stats_df[stats_df["Class"] == cls_name]
        ax.plot(subset["Level"], subset[metric], "o-",
                color=CLASS_COLORS[cls_name], label=cls_name, linewidth=2, markersize=8)
    ax.set_xlabel("Encoder Level")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.set_xticks(range(nb_levels))
    ax.legend()

plt.suptitle("Feature Map Statistics Across Encoder Levels", fontsize=14)
plt.tight_layout()
plt.show()

## 12. Codebook Distributions Across Diagnosis Codes

Detailed distributional analysis of how codebook usage differs between diagnostic groups (AD, CN, MCI).

- **12a**: Per-class mean codebook distributions with JS-divergence between every class pair
- **12b**: Per-code statistical testing (Kruskal-Wallis) to find codes with significantly different usage across groups
- **12c**: Violin plots for the top discriminative codes showing the full per-subject distribution by class
- **12d**: Class-pair divergence profiles — which codebook entries contribute most to each pairwise divergence

### 12a. Jensen-Shannon Divergence Between Class Codebook Distributions

For each level, compute the mean codebook usage distribution per class, then measure the JS-divergence between every class pair. JSD is bounded [0, 1] (using log base 2) — higher values indicate the two classes use the codebook more differently.

In [ ]:
from scipy.spatial.distance import jensenshannon
from itertools import combinations

for lvl in range(nb_levels):
    # Mean codebook distribution per class (add small epsilon for numerical stability)
    eps = 1e-12
    class_dists = {}
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        mean_hist = all_histograms[lvl][mask].mean(axis=0) + eps
        mean_hist /= mean_hist.sum()  # re-normalise
        class_dists[cls_name] = mean_hist

    # JSD matrix
    n_cls = len(CLASS_NAMES)
    jsd_matrix = np.zeros((n_cls, n_cls))
    for (i, name_i), (j, name_j) in combinations(enumerate(CLASS_NAMES), 2):
        jsd = jensenshannon(class_dists[name_i], class_dists[name_j], base=2) ** 2
        jsd_matrix[i, j] = jsd
        jsd_matrix[j, i] = jsd

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"Level {lvl} — Class Codebook Distributions & JSD", fontsize=14)

    # Left: overlaid distributions
    ax = axes[0]
    for cls_name, dist in class_dists.items():
        ax.plot(dist, alpha=0.7, label=cls_name, color=CLASS_COLORS[cls_name], linewidth=1.2)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mean frequency")
    ax.set_title("Mean codebook distribution per class")
    ax.legend()

    # Right: JSD heatmap
    ax = axes[1]
    im = ax.imshow(jsd_matrix, cmap="YlOrRd", vmin=0)
    ax.set_xticks(range(n_cls))
    ax.set_yticks(range(n_cls))
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticklabels(CLASS_NAMES)
    for i in range(n_cls):
        for j in range(n_cls):
            ax.text(j, i, f"{jsd_matrix[i, j]:.4f}", ha="center", va="center",
                    color="white" if jsd_matrix[i, j] > jsd_matrix.max() * 0.6 else "black")
    ax.set_title("Jensen-Shannon Divergence (squared)")
    fig.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()
    plt.show()

    # Print summary
    for (i, ni), (j, nj) in combinations(enumerate(CLASS_NAMES), 2):
        print(f"  Level {lvl} JSD²({ni}, {nj}) = {jsd_matrix[i, j]:.6f}")

### 12b. Per-Code Statistical Testing (Kruskal-Wallis)

For each codebook entry, test whether its frequency distribution differs significantly across the three diagnostic groups using the non-parametric Kruskal-Wallis H-test (no normality assumption). Results are corrected for multiple comparisons (Benjamini-Hochberg FDR).

In [ ]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

TOP_K_CODES = 20  # codes to highlight per level

kw_results = {}  # lvl -> DataFrame of results

for lvl in range(nb_levels):
    h_stats, p_vals, code_indices = [], [], []

    for code_idx in range(nb_entries):
        groups = [all_histograms[lvl][labels == c, code_idx] for c in range(len(CLASS_NAMES))]
        # Skip codes that are never used (all zeros)
        if all(g.sum() == 0 for g in groups):
            continue
        stat, p = kruskal(*groups)
        h_stats.append(stat)
        p_vals.append(p)
        code_indices.append(code_idx)

    # FDR correction
    reject, p_adj, _, _ = multipletests(p_vals, method="fdr_bh", alpha=0.05)

    df = pd.DataFrame({
        "code": code_indices,
        "H_stat": h_stats,
        "p_value": p_vals,
        "p_adj": p_adj,
        "significant": reject,
    }).sort_values("H_stat", ascending=False).reset_index(drop=True)

    # Add per-class mean frequencies for context
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        df[f"mean_{cls_name}"] = df["code"].apply(lambda c: all_histograms[lvl][mask, c].mean())

    kw_results[lvl] = df

    n_sig = df["significant"].sum()
    n_tested = len(df)
    print(f"Level {lvl}: {n_sig}/{n_tested} codes significant (FDR < 0.05)")

    # Plot: H-statistic across all codes (highlight significant)
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f"Level {lvl} — Kruskal-Wallis Test per Codebook Entry", fontsize=14)

    ax = axes[0]
    colors = ["#e74c3c" if s else "#95a5a6" for s in df["significant"]]
    ax.bar(range(len(df)), df["H_stat"], color=colors, width=1.0)
    ax.set_xlabel("Code (sorted by H-statistic)")
    ax.set_ylabel("Kruskal-Wallis H")
    ax.set_title(f"H-statistic (red = FDR < 0.05, {n_sig} significant)")

    # Right: volcano-style — H-stat vs -log10(p_adj)
    ax = axes[1]
    neg_log_p = -np.log10(df["p_adj"].clip(lower=1e-300))
    ax.scatter(df["H_stat"], neg_log_p, c=colors, s=12, alpha=0.7)
    ax.axhline(-np.log10(0.05), color="gray", linestyle="--", linewidth=0.8, label="FDR = 0.05")
    ax.set_xlabel("Kruskal-Wallis H")
    ax.set_ylabel("-log10(adjusted p-value)")
    ax.set_title("Volcano plot")
    ax.legend()

    plt.tight_layout()
    plt.show()

    # Print top codes
    print(f"\n  Top {TOP_K_CODES} most discriminative codes (by H-stat):")
    top = df.head(TOP_K_CODES)
    for _, row in top.iterrows():
        sig_marker = "*" if row["significant"] else " "
        means = " | ".join(f"{cls}: {row[f'mean_{cls}']:.4f}" for cls in CLASS_NAMES)
        print(f"  {sig_marker} Code {int(row['code']):>3d}  H={row['H_stat']:7.2f}  "
              f"p_adj={row['p_adj']:.2e}  [{means}]")
    print()

### 12c. Violin Plots for Top Discriminative Codes

For the top-K most discriminative codes (by Kruskal-Wallis H-stat) at each level, show violin + strip plots of the per-subject frequency distributions broken down by class. This reveals whether the difference is in central tendency, spread, or shape.

In [ ]:
TOP_K_VIOLIN = 8  # codes to plot per level

for lvl in range(nb_levels):
    df = kw_results[lvl]
    top_codes = df.head(TOP_K_VIOLIN)["code"].values.astype(int)

    n_codes = len(top_codes)
    fig, axes = plt.subplots(2, (n_codes + 1) // 2, figsize=(4 * ((n_codes + 1) // 2), 10))
    axes = axes.flatten()
    fig.suptitle(f"Level {lvl} — Top {n_codes} Discriminative Code Distributions", fontsize=14)

    for ax_idx, code_idx in enumerate(top_codes):
        ax = axes[ax_idx]

        # Build a long-form DataFrame for seaborn
        plot_data = []
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            mask = labels == cls_idx
            freqs = all_histograms[lvl][mask, code_idx]
            for f in freqs:
                plot_data.append({"Class": cls_name, "Frequency": f})
        plot_df = pd.DataFrame(plot_data)

        sns.violinplot(
            data=plot_df, x="Class", y="Frequency", ax=ax,
            palette=CLASS_COLORS, inner=None, alpha=0.3, linewidth=0.8,
            order=CLASS_NAMES,
        )
        sns.stripplot(
            data=plot_df, x="Class", y="Frequency", ax=ax,
            palette=CLASS_COLORS, size=2, alpha=0.4, jitter=True,
            order=CLASS_NAMES,
        )

        row = df[df["code"] == code_idx].iloc[0]
        sig = "***" if row["p_adj"] < 0.001 else "**" if row["p_adj"] < 0.01 else "*" if row["p_adj"] < 0.05 else "ns"
        ax.set_title(f"Code {code_idx} ({sig})\nH={row['H_stat']:.1f}", fontsize=10)
        ax.set_xlabel("")

    # Hide unused axes
    for ax_idx in range(n_codes, len(axes)):
        axes[ax_idx].set_visible(False)

    plt.tight_layout()
    plt.show()

### 12d. Class-Pair Divergence Profiles

For each pair of diagnostic classes, compute the per-entry contribution to the overall JS-divergence. This shows *which specific codebook entries* drive the distributional difference between any two classes — useful for identifying disease-specific encoding patterns.

In [ ]:
def per_entry_jsd(p, q):
    """Compute per-bin contribution to JS-divergence (sums to JSD)."""
    eps = 1e-12
    p = np.asarray(p, dtype=np.float64) + eps
    q = np.asarray(q, dtype=np.float64) + eps
    p /= p.sum()
    q /= q.sum()
    m = 0.5 * (p + q)
    # Per-entry: 0.5 * [p_i * log(p_i/m_i) + q_i * log(q_i/m_i)]
    contrib = 0.5 * (p * np.log2(p / m) + q * np.log2(q / m))
    return contrib  # (nb_entries,), sums to JSD²


class_pairs = list(combinations(range(len(CLASS_NAMES)), 2))
pair_names = [(CLASS_NAMES[i], CLASS_NAMES[j]) for i, j in class_pairs]
pair_colors = ["#8e44ad", "#e67e22", "#16a085"]  # one color per pair

TOP_N_ENTRIES = 10  # top entries to annotate per pair

for lvl in range(nb_levels):
    eps = 1e-12

    # Mean distributions per class
    dists = {}
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        d = all_histograms[lvl][labels == cls_idx].mean(axis=0) + eps
        dists[cls_idx] = d / d.sum()

    fig, axes = plt.subplots(len(class_pairs), 1, figsize=(16, 4.5 * len(class_pairs)),
                              sharex=True)
    if len(class_pairs) == 1:
        axes = [axes]
    fig.suptitle(f"Level {lvl} — Per-Entry JSD Contribution by Class Pair", fontsize=14, y=1.01)

    for ax, (ci, cj), (ni, nj), col in zip(axes, class_pairs, pair_names, pair_colors):
        contrib = per_entry_jsd(dists[ci], dists[cj])

        ax.bar(range(nb_entries), contrib, color=col, alpha=0.7, width=1.0)
        ax.set_ylabel("JSD contribution")
        ax.set_title(f"{ni} vs {nj}  (total JSD² = {contrib.sum():.6f})")

        # Annotate top entries
        top_idx = np.argsort(contrib)[-TOP_N_ENTRIES:][::-1]
        for rank, idx in enumerate(top_idx):
            if rank < 5:  # annotate top 5 only to avoid clutter
                ax.annotate(
                    f"{idx}", (idx, contrib[idx]),
                    textcoords="offset points", xytext=(0, 6),
                    fontsize=7, ha="center", color=col,
                )

    axes[-1].set_xlabel("Codebook entry")
    plt.tight_layout()
    plt.show()

    # Summary table: top entries per pair
    print(f"Level {lvl} — Top {TOP_N_ENTRIES} entries driving each class-pair divergence:")
    for (ci, cj), (ni, nj) in zip(class_pairs, pair_names):
        contrib = per_entry_jsd(dists[ci], dists[cj])
        top_idx = np.argsort(contrib)[-TOP_N_ENTRIES:][::-1]
        entries_str = ", ".join(
            f"{idx}({contrib[idx]:.5f})" for idx in top_idx
        )
        print(f"  {ni} vs {nj}: {entries_str}")
    print()